# Jianing — TF, full-text MiniLM, and their fusion + Linear SVM
Six steps: **audit → split → TF → chunked MiniLM + fusion → train → compare AUROC**.

AI = 1; human = 0. Only `train.csv` is used in this supervised experiment. `train_starter_2000.csv` is a subset, `unlabeled.csv` is for a later semi-supervised experiment, and `test.csv`/`sample_submission.csv` are for final prediction/submission.

Run top to bottom once. To change C in the same active kernel, edit **5A**, then run **5A → 5B → 6**. All training models are rebuilt; feature matrices are reused. New results receive a separate run folder.

AI assistance: Codex helped implement and execute this notebook. Jianing should inspect, rerun, and interpret the experiments.

## 1. Check the raw data
ID identifies an article, text is the input, and label is the answer. ID is not a predictive feature. Show only compact statistics; full passages are not printed. Exact and normalized duplicate checks do not rule out paraphrases or shared sources.

In [ ]:
from pathlib import Path
import os, json, hashlib, time, warnings, sys
from datetime import datetime, timezone
import importlib.metadata as metadata

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/raw/train.csv").exists() and (p / "notebooks").exists()), None)
assert ROOT is not None, "Open this notebook from the kaggle project directory."
os.environ["HF_HOME"] = str(ROOT / ".cache/huggingface")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["MPLCONFIGDIR"] = str(ROOT / ".cache/matplotlib")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits

SEED = 42
DATA = ROOT / "data/processed/jianing_svm"
RESULTS = ROOT / "results/jianing_svm"
DATA.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
train_path = ROOT / "data/raw/train.csv"
df = pd.read_csv(train_path)
assert set(df.columns) == {"id", "text", "label"} and len(df) == 10000
assert not df.isna().any().any() and df.id.is_unique
assert set(df.label.unique()) == {0, 1}
assert df.text.map(lambda t: isinstance(t, str) and bool(t.strip())).all()
normalized_text = df.text.str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
word_lengths = df.text.str.split().str.len()
audit = {
    "rows": len(df), "ID dtype": str(df.id.dtype), "unique IDs": df.id.nunique(),
    "missing cells": int(df.isna().sum().sum()),
    "exact duplicate texts": int(df.text.duplicated().sum()),
    "normalized duplicate texts": int(normalized_text.duplicated().sum()),
    "text words: minimum": int(word_lengths.min()),
    "text words: median": float(word_lengths.median()),
    "text words: maximum": int(word_lengths.max()),
}
print("Example IDs:", df.id.head(5).tolist())
display(pd.Series(audit, name="Raw data summary").to_frame())
class_counts = df.label.value_counts().sort_index()
display(pd.DataFrame({"class": ["Human (0)", "AI (1)"],
                      "count": class_counts.to_numpy(), "percent": class_counts.to_numpy()/len(df)*100}))
DATA_SHA = hashlib.sha256(train_path.read_bytes()).hexdigest()
(DATA / "data_audit.json").write_text(json.dumps({**audit, "train_sha256": DATA_SHA}, indent=2))

## 2. Split: 8,000 training / 2,000 validation
`random_state=42` fixes the draw; `stratify=df.label` preserves the label proportions. Every teammate must use the same saved IDs. We recompute the requested split and check it against any existing shared file; a mismatch stops execution instead of silently changing the team's split.

In [ ]:
train_ids, val_ids = train_test_split(
    df.id.to_numpy(), test_size=2000, random_state=42, stratify=df.label.to_numpy()
)
split = pd.concat([
    pd.DataFrame({"id": train_ids, "partition": "train"}),
    pd.DataFrame({"id": val_ids, "partition": "validation"})
], ignore_index=True)
SPLIT_PATH = ROOT / "data/splits/seed42_stratified_8000_2000.csv"
SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)
if SPLIT_PATH.exists():
    existing = pd.read_csv(SPLIT_PATH)
    assert existing.id.is_unique and len(existing) == len(split)
    assert existing.set_index("id").partition.to_dict() == split.set_index("id").partition.to_dict(), \
        "Different team split detected. Agree on shared IDs before comparing models."
else:
    split.to_csv(SPLIT_PATH, index=False)
indexed = df.set_index("id", drop=False)
fit = indexed.loc[train_ids].reset_index(drop=True)
val = indexed.loc[val_ids].reset_index(drop=True)
y_fit, y_val = fit.label.to_numpy(), val.label.to_numpy()
assert set(fit.id).isdisjoint(set(val.id))
assert fit.label.value_counts().to_dict() == {0:4000, 1:4000}
assert val.label.value_counts().to_dict() == {0:1000, 1:1000}
canon = lambda s: set(s.str.lower().str.replace(r"\s+", " ", regex=True).str.strip())
assert not (canon(fit.text) & canon(val.text)), "Resolve duplicate leakage before training."
summary = pd.DataFrame([
    {"partition": name, "rows": len(d), "human": int((d.label==0).sum()),
     "AI": int((d.label==1).sum()), "AI_percent": 100*d.label.mean()}
    for name,d in [("Original",df),("Train",fit),("Validation",val)]
])
display(summary)
summary.to_csv(DATA / "split_summary.csv", index=False)
fit[["id","label"]].to_csv(DATA / "train_rows.csv", index=False)
val[["id","label"]].to_csv(DATA / "validation_rows.csv", index=False)
SPLIT_SHA = hashlib.sha256(SPLIT_PATH.read_bytes()).hexdigest()
print("No shared IDs or normalized duplicate text across the two partitions.")

## 3. Preprocessing A — Term frequency (TF)
Count words, then L2-normalize each document vector. This is not TF-IDF: no IDF weighting is used. Lowercase; keep numbers and stop words; no lemmatization. The tokenizer omits punctuation and single-character tokens. `min_df=5` excludes words appearing in fewer than five training articles; this is a practical preset, not a tuned optimum.

Fit the vocabulary only on training text. Save sparse matrices (mostly zeros) and the vocabulary. Matrix rows follow `train_rows.csv` / `validation_rows.csv`.

In [ ]:
TF_CONFIG = {"lowercase": True, "min_df": 5, "max_df": 1.0,
             "token_pattern": r"(?u)\b\w\w+\b", "ngram_range": [1,1], "norm": "l2"}
t0 = time.perf_counter()
vectorizer = CountVectorizer(**{k:(tuple(v) if k=="ngram_range" else v)
                              for k,v in TF_CONFIG.items() if k!="norm"})
counts_fit = vectorizer.fit_transform(fit.text)
X_tf_fit = normalize(counts_fit, norm="l2")
tf_train_seconds = time.perf_counter()-t0
t0 = time.perf_counter()
X_tf_val = normalize(vectorizer.transform(val.text), norm="l2")
tf_val_seconds = time.perf_counter()-t0
sp.save_npz(DATA / "tf_train.npz", X_tf_fit)
sp.save_npz(DATA / "tf_validation.npz", X_tf_val)
vocabulary = pd.DataFrame({"column": np.arange(X_tf_fit.shape[1]),
    "word": vectorizer.get_feature_names_out(),
    "training_document_count": np.asarray((counts_fit>0).sum(axis=0)).ravel()})
vocabulary.to_csv(DATA / "tf_vocabulary.csv", index=False)
tf_summary = pd.DataFrame([
    {"partition": name, "rows": x.shape[0], "features": x.shape[1],
     "nonzero_percent": 100*x.nnz/(x.shape[0]*x.shape[1]),
     "empty_feature_rows": int((x.getnnz(axis=1)==0).sum()), "feature_seconds": seconds}
    for name,x,seconds in [("Train",X_tf_fit,tf_train_seconds),("Validation",X_tf_val,tf_val_seconds)]
])
display(tf_summary.round(4))
print("Ten most frequent training-vocabulary words:")
display(vocabulary.nlargest(10,"training_document_count")[["word","training_document_count"]].reset_index(drop=True))
tf_summary.to_csv(DATA / "tf_summary.csv", index=False)

## 4. Preprocessing B — MiniLM with full-text chunking
Use a frozen `all-MiniLM-L6-v2` encoder. Tokenize the complete article without truncation; split its token IDs into nonoverlapping chunks of at most **254 content tokens**, reserving two positions for special tokens. Feed those IDs directly to the encoder: no decoding/re-tokenization that might silently change coverage.

Each chunk becomes a normalized 384-dimensional vector. Average chunk vectors weighted by their content-token counts, then normalize the article vector. Every token is processed, but different chunks cannot attend to each other and averaging can dilute local signals. This is a first full-text aggregation method, not a guarantee of better performance.

The cache is reused only when data, row order, encoder revision, preprocessing and relevant library versions match. The printed summary reports coverage and chunk counts.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

EMB_CONFIG = {
    "model": "sentence-transformers/all-MiniLM-L6-v2",
    "revision": "1110a243fdf4706b3f48f1d95db1a4f5529b4d41",
    "max_sequence_length": 256, "overlap": 0,
    "aggregation": "content_token_weighted_mean_of_unit_chunk_vectors_then_L2",
    "batch_size": 32, "device": "cpu"
}
FORCE_REBUILD_EMBEDDINGS = False
VERSIONS = {k: metadata.version(k) for k in ["numpy","pandas","scipy","scikit-learn",
    "torch","transformers","sentence-transformers","tokenizers"]}
cache_spec = {"data_sha":DATA_SHA, "train_ids":fit.id.tolist(), "val_ids":val.id.tolist(),
              "embedding":EMB_CONFIG, "versions":VERSIONS}
CACHE_KEY = hashlib.sha256(json.dumps(cache_spec,sort_keys=True).encode()).hexdigest()
cache_meta = DATA / "minilm_cache.json"
cache_files = [DATA / f"minilm_{s}.npy" for s in ["train","validation"]]
cache_files += [DATA / "minilm_chunks.csv"]
cache_valid = (not FORCE_REBUILD_EMBEDDINGS and cache_meta.exists()
               and all(f.exists() for f in cache_files)
               and json.loads(cache_meta.read_text())["cache_key"] == CACHE_KEY)

if cache_valid:
    X_emb_fit = np.load(DATA / "minilm_train.npy", allow_pickle=False)
    X_emb_val = np.load(DATA / "minilm_validation.npy", allow_pickle=False)
    chunk_audit = pd.read_csv(DATA / "minilm_chunks.csv")
    embedding_timing = json.loads(cache_meta.read_text())["original_encoding_seconds"]
    print("Loaded matching full-text embeddings from cache; times below refer to their original computation.")
else:
    torch.manual_seed(SEED)
    torch.set_num_threads(4)
    encoder = SentenceTransformer(EMB_CONFIG["model"], revision=EMB_CONFIG["revision"],
        cache_folder=str(ROOT / ".cache/huggingface/hub"), device=EMB_CONFIG["device"])
    encoder.max_seq_length = EMB_CONFIG["max_sequence_length"]
    encoder.eval()
    tokenizer = encoder.tokenizer
    capacity = encoder.max_seq_length-tokenizer.num_special_tokens_to_add(pair=False)
    assert capacity == 254
    # This pinned BERT tokenizer uses [CLS] content [SEP]. Verify its template.
    probe = "Check the chunk boundary."
    assert tokenizer(probe, add_special_tokens=True)["input_ids"] == (
        [tokenizer.cls_token_id] + tokenizer(probe, add_special_tokens=False)["input_ids"]
        + [tokenizer.sep_token_id])

    def encode_full_text(frame, partition):
        start_time = time.perf_counter()
        tokens = tokenizer(frame.text.tolist(), add_special_tokens=False,
                           truncation=False, padding=False, verbose=False)["input_ids"]
        assert all(len(t)>0 for t in tokens)
        chunks, owners, weights, audits = [], [], [], []
        for i, ids in enumerate(tokens):
            pieces = [ids[j:j+capacity] for j in range(0,len(ids),capacity)]
            # Exact coverage, no dropped/repeated content tokens.
            assert [token for piece in pieces for token in piece] == ids
            audits.append({"id":frame.id.iloc[i], "partition":partition,
                           "content_tokens":len(ids), "processed_tokens":sum(map(len,pieces)),
                           "chunks":len(pieces), "dropped_tokens":0})
            for piece in pieces:
                prepared = [tokenizer.cls_token_id] + piece + [tokenizer.sep_token_id]
                assert len(prepared)<=encoder.max_seq_length
                chunks.append({"input_ids":prepared,"attention_mask":[1]*len(prepared)})
                owners.append(i);weights.append(len(piece))
        sums = np.zeros((len(frame),384),dtype=np.float64)
        owners,weights = np.asarray(owners),np.asarray(weights)
        for start in tqdm(range(0,len(chunks),EMB_CONFIG["batch_size"]), desc=f"Encoding {partition} chunks", mininterval=15):
            stop = min(start+EMB_CONFIG["batch_size"],len(chunks))
            batch = tokenizer.pad(chunks[start:stop],padding=True,return_tensors="pt")
            assert batch["input_ids"].shape[1]<=encoder.max_seq_length
            batch = {k:v.to(EMB_CONFIG["device"]) for k,v in batch.items()}
            with torch.inference_mode():
                vectors = encoder(batch)["sentence_embedding"]
                vectors = torch.nn.functional.normalize(vectors,p=2,dim=1).cpu().numpy()
            np.add.at(sums,owners[start:stop],vectors*weights[start:stop,None])
        means = sums / np.asarray([len(t) for t in tokens])[:,None]
        assert (np.linalg.norm(means,axis=1)>0).all()
        return normalize(means).astype(np.float32), pd.DataFrame(audits), time.perf_counter()-start_time

    X_emb_fit, audit_fit, train_seconds = encode_full_text(fit,"Train")
    X_emb_val, audit_val, val_seconds = encode_full_text(val,"Validation")
    chunk_audit = pd.concat([audit_fit,audit_val],ignore_index=True)
    embedding_timing = {"train":train_seconds,"validation":val_seconds}
    np.save(DATA / "minilm_train.npy",X_emb_fit)
    np.save(DATA / "minilm_validation.npy",X_emb_val)
    chunk_audit.to_csv(DATA / "minilm_chunks.csv",index=False)
    cache_meta.write_text(json.dumps({"cache_key":CACHE_KEY,"config":EMB_CONFIG,
        "original_encoding_seconds":embedding_timing,"versions":VERSIONS},indent=2))

assert X_emb_fit.shape==(8000,384) and X_emb_val.shape==(2000,384)
assert np.isfinite(X_emb_fit).all() and np.isfinite(X_emb_val).all()
assert (chunk_audit.content_tokens==chunk_audit.processed_tokens).all()
assert chunk_audit.dropped_tokens.sum()==0
emb_summary = pd.DataFrame([
    {"partition":name,"rows":len(x),"features":x.shape[1],"total_chunks":int(a.chunks.sum()),
     "multi_chunk_articles":int((a.chunks>1).sum()),"max_chunks_per_article":int(a.chunks.max()),
     "dropped_tokens":int(a.dropped_tokens.sum()),"feature_seconds":embedding_timing[key]}
    for name,key,x in [("Train","train",X_emb_fit),("Validation","validation",X_emb_val)]
    for a in [chunk_audit[chunk_audit.partition==name]]
])
display(emb_summary.round(3))
emb_summary.to_csv(DATA / "minilm_summary.csv",index=False)
FEATURE_SPEC = {"data_sha256":DATA_SHA,"split_sha256":SPLIT_SHA,"TF":TF_CONFIG,
                "MiniLM_chunked":EMB_CONFIG,"embedding_cache_key":CACHE_KEY}
(DATA / "feature_config.json").write_text(json.dumps(FEATURE_SPEC,indent=2))
print("Full token coverage verified; all processed features saved to data/processed/jianing_svm.")

### 4C. Preprocessing C — TF + MiniLM feature fusion
For each article, concatenate its TF vector and full-text MiniLM vector. Both blocks already have unit L2 norm; use equal weights and L2-normalize the concatenated vector so the combined input also has unit norm. No feature block weighting is tuned in this experiment.

This gives **14,198 + 384 = 14,582 columns** for the current vocabulary. The same article IDs and row order are preserved. This is feature fusion before one SVM, not averaging two classifiers' predictions. Hypothesis: word usage and pretrained text information may be complementary; better performance is not guaranteed.

In [ ]:
FUSION_CONFIG = {"TF_weight": 1.0, "MiniLM_weight": 1.0,
                 "column_order": ["TF", "MiniLM_chunked"], "final_normalization": "l2"}
assert X_tf_fit.shape[0] == X_emb_fit.shape[0] == len(fit)
assert X_tf_val.shape[0] == X_emb_val.shape[0] == len(val)

def fuse_features(tf_matrix, embeddings):
    combined = sp.hstack([tf_matrix * FUSION_CONFIG["TF_weight"],
                          sp.csr_matrix(embeddings) * FUSION_CONFIG["MiniLM_weight"]], format="csr")
    return normalize(combined, norm="l2", copy=False)

t0 = time.perf_counter()
X_fused_fit = fuse_features(X_tf_fit, X_emb_fit)
fusion_train_seconds = time.perf_counter() - t0
t0 = time.perf_counter()
X_fused_val = fuse_features(X_tf_val, X_emb_val)
fusion_val_seconds = time.perf_counter() - t0
assert X_fused_fit.shape[1] == X_tf_fit.shape[1] + X_emb_fit.shape[1]
for matrix in [X_fused_fit, X_fused_val]:
    assert np.isfinite(matrix.data).all()
    assert np.allclose(np.asarray(matrix.multiply(matrix).sum(axis=1)).ravel(), 1, atol=1e-6)
sp.save_npz(DATA / "fused_train.npz", X_fused_fit)
sp.save_npz(DATA / "fused_validation.npz", X_fused_val)
fusion_summary = pd.DataFrame([
    {"partition": name, "rows": x.shape[0], "TF_features": X_tf_fit.shape[1],
     "MiniLM_features": X_emb_fit.shape[1], "combined_features": x.shape[1],
     "concatenation_seconds": seconds}
    for name,x,seconds in [("Train", X_fused_fit, fusion_train_seconds),
                           ("Validation", X_fused_val, fusion_val_seconds)]
])
display(fusion_summary.round(4))
fusion_summary.to_csv(DATA / "fusion_summary.csv", index=False)
FEATURE_SPEC["TF_MiniLM_fused"] = FUSION_CONFIG.copy()
(DATA / "feature_config.json").write_text(json.dumps(FEATURE_SPEC, indent=2))

## 5. Train Linear SVM
### 5A. Edit C here
C controls the penalty for training errors relative to weight regularization. Smaller C means stronger regularization. Start with a coarse logarithmic grid; these are candidates, not known optima. Change any list below, then run **5A, 5B, and 6**. Do not rerun feature extraction just to change C.

Keep the validation split and features fixed. Each run gets a separate output folder so earlier results are preserved.

In [ ]:
# EDIT THESE LISTS to try different C values.
C_BY_METHOD = {
    "TF": [0.01, 0.1, 1.0, 10.0, 100.0],
    "MiniLM_chunked": [0.01, 0.1, 1.0, 10.0, 100.0],
    "TF_MiniLM_fused": [0.01, 0.1, 1.0, 10.0, 100.0],
}
MAX_ITER = 50000
TOL = 1e-5
for values in C_BY_METHOD.values():
    assert values and all(np.isfinite(c) and c>0 for c in values)
    assert len(values)==len(set(values)), "Remove duplicate C values."
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN = RESULTS / RUN_ID
RUN.mkdir(parents=True,exist_ok=False)
RUN_CONFIG = {"run_id":RUN_ID,"seed":SEED,"C_by_method":C_BY_METHOD,
              "svm":{"dual":"auto","max_iter":MAX_ITER,"tol":TOL},
              "features":FEATURE_SPEC}
(RUN / "config.json").write_text(json.dumps(RUN_CONFIG,indent=2))
print("Candidates:",C_BY_METHOD)
print("Run ID:",RUN_ID)

### 5B. Fit each combination
Only training features and labels enter `.fit()`. Record fitting time and convergence. All models are rebuilt on each execution; a convergence failure stops the run rather than reporting unchecked scores.

In [ ]:
feature_sets = {"TF":(X_tf_fit,X_tf_val),"MiniLM_chunked":(X_emb_fit,X_emb_val),
                "TF_MiniLM_fused":(X_fused_fit,X_fused_val)}
trained_models, training_records = {}, []
for method,values in C_BY_METHOD.items():
    X_train,_ = feature_sets[method]
    for C in values:
        model = LinearSVC(C=C,dual="auto",max_iter=MAX_ITER,tol=TOL,random_state=SEED)
        started = time.perf_counter()
        with threadpool_limits(limits=4), warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            model.fit(X_train,y_fit)
        seconds = time.perf_counter()-started
        for warning in caught:
            if issubclass(warning.category,ConvergenceWarning):
                raise RuntimeError(f"{method}, C={C}: did not converge; raise MAX_ITER and rerun 5A–6.")
            warnings.warn(str(warning.message),warning.category)
        assert model.classes_.tolist()==[0,1]
        trained_models[(method,C)] = model
        training_records.append({"representation":method,"C":C,"svm_fit_seconds":seconds,
                                 "iterations":int(np.max(model.n_iter_)),"converged":True})
        print(f"Fitted {method:16s} C={C:g} in {seconds:.3f}s")
pd.DataFrame(training_records).to_csv(RUN / "training_log.csv",index=False)
print("Training complete. Run step 6 to calculate and save AUROC.")

## 6. Calculate AUROC and compare all combinations
Use continuous SVM decision scores, not hard labels. Higher scores indicate AI; these scores are not probabilities. AUROC measures ranking, not percentage classified correctly. A large train–validation gap suggests a generalization problem but does not identify its cause.

One table contains every representation/C combination. `svm_fit_seconds` excludes feature extraction; `train_pipeline_seconds` adds training-feature preparation. Cached embeddings retain their original extraction time for an honest cost comparison; they were not recomputed separately for every C.

The supplied report lists **0.90 as a provisional private-Kaggle full-credit threshold**, subject to instructor confirmation. It is not a required local validation score. Local validation cannot establish private-test performance. The report also requires at least three approaches and tuning records for at least two approaches across the team.

Fusion tests whether lexical and embedding information are complementary. Compare at the same C as well as the best searched C per representation. Fusion preparation cost includes both representations and concatenation, even when existing features are reused. This remains three representations with the same classifier family, not three different classifier algorithms.

In [ ]:
rows = []
validation_scores = val[["id","label"]].rename(columns={"label":"true_label"}).copy()
for record in training_records:
    method,C = record["representation"],record["C"]
    X_train,X_valid = feature_sets[method]
    model = trained_models[(method,C)]
    train_score = model.decision_function(X_train)
    val_score = model.decision_function(X_valid)
    assert np.isfinite(train_score).all() and np.isfinite(val_score).all()
    train_auc = roc_auc_score(y_fit,train_score)
    val_auc = roc_auc_score(y_val,val_score)
    prep_train, prep_val = {
        "TF": (tf_train_seconds, tf_val_seconds),
        "MiniLM_chunked": (embedding_timing["train"], embedding_timing["validation"]),
        "TF_MiniLM_fused": (tf_train_seconds + embedding_timing["train"] + fusion_train_seconds,
                            tf_val_seconds + embedding_timing["validation"] + fusion_val_seconds)
    }[method]
    rows.append({**record,"train_rows":len(y_fit),"validation_rows":len(y_val),
        "features":X_train.shape[1],"train_AUROC":train_auc,"validation_AUROC":val_auc,
        "train_val_gap":train_auc-val_auc,"feature_train_seconds":prep_train,
        "feature_validation_seconds":prep_val,
        "train_pipeline_seconds":prep_train+record["svm_fit_seconds"]})
    validation_scores[f"{method}__C_{C:g}__decision_score"] = val_score
comparison = pd.DataFrame(rows).sort_values(["representation","C"]).reset_index(drop=True)
comparison["best_in_method"] = False
for method in C_BY_METHOD:
    winner = comparison[comparison.representation==method].sort_values(
        ["validation_AUROC","C"],ascending=[False,True]).index[0]
    comparison.loc[winner,"best_in_method"] = True
columns = ["representation","C","features","train_rows","validation_rows","train_AUROC",
           "validation_AUROC","train_val_gap","svm_fit_seconds","feature_train_seconds",
           "feature_validation_seconds","train_pipeline_seconds","iterations","converged","best_in_method"]
comparison = comparison[columns]
with pd.option_context("display.max_columns",None,"display.width",220):
    display(comparison.round(6))
comparison.to_csv(RUN / "comparison.csv",index=False)
validation_scores.to_csv(RUN / "validation_scores.csv",index=False)

fig,ax = plt.subplots(figsize=(6,5))
for r in comparison[comparison.best_in_method].itertuples():
    s = validation_scores[f"{r.representation}__C_{r.C:g}__decision_score"]
    fpr,tpr,_ = roc_curve(y_val,s)
    ax.plot(fpr,tpr,label=f"{r.representation}, C={r.C:g}: {r.validation_AUROC:.4f}")
    print(f"Best {r.representation}: C={r.C:g}, train={r.train_AUROC:.6f}, validation={r.validation_AUROC:.6f}")
    if r.C in [min(C_BY_METHOD[r.representation]),max(C_BY_METHOD[r.representation])]:
        print("  Search-boundary winner: the optimum outside the grid is unknown.")
ax.plot([0,1],[0,1],"k--",linewidth=1)
ax.set(xlabel="False positive rate",ylabel="True positive rate",title="Validation ROC: selected C per method")
ax.legend(loc="lower right");fig.tight_layout()
fig.savefig(RUN / "validation_roc.png",dpi=160)
plt.show()

manifest = {**RUN_CONFIG,"versions":VERSIONS,"python":sys.version.split()[0],
    "embedding_loaded_from_cache":bool(cache_valid),
    "feature_time_note":"Encoding times refer to original feature computation; encoder load/download excluded.",
    "score_type":"Uncalibrated decision score, not probability",
    "limitations":["One validation split used for C selection; no independent test estimate",
                   "No exhaustive source/near-duplicate audit",
                   "Chunk pooling loses cross-chunk context; no token truncation"],
    "scope":"Supervised model comparison only; team still needs report Q2–Q5 and final calibrated predictions"}
(RUN / "run_manifest.json").write_text(json.dumps(manifest,indent=2))
for f in ["data_audit.json","split_summary.csv","tf_summary.csv","minilm_summary.csv","fusion_summary.csv","feature_config.json"]:
    (RUN / f).write_bytes((DATA / f).read_bytes())
# Snapshot the actual feature row order, independent of later experiments.
for f in ["train_rows.csv","validation_rows.csv"]:
    (RUN / f).write_bytes((DATA / f).read_bytes())
(ROOT / "configs/jianing_svm.json").write_text(json.dumps(RUN_CONFIG,indent=2))
(RESULTS / "latest_run.txt").write_text(RUN_ID+"\n")
print("Saved this run to:",RUN.relative_to(ROOT))
print("Saved processed features to:",DATA.relative_to(ROOT))
print("All previous run folders are retained. No Kaggle submission has been generated.")

Sources: [LinearSVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html), [AUROC](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html), [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2), and the supplied course report template. Final probability calibration, if SVM is selected, must be trained using training-only internal data and evaluated before final refitting.

## 7. Supplementary optimization — features and controlled fusion
The original six steps remain above. Our previous best was TF + chunked MiniLM, C=1 (validation AUROC 0.922313; public Kaggle 0.84249). We now test better representations while keeping the same labeled split and Linear SVM family.

- **TF-IDF unigrams:** isolates IDF weighting relative to original TF.
- **Word unigrams + bigrams:** includes short phrases; logarithmic TF dampens repeated words.
- **Character 3–5 grams:** includes short character sequences, case and punctuation patterns. It may capture style but can also learn topic/source artifacts.
- **Weighted fusion:** tests reduced MiniLM weights, rather than assuming equal weights are best.

Vocabulary/IDF are learned only from the 8,000 training articles. No test data or Kaggle feedback is used to select new settings. This is a bounded 30-candidate search. The winning validation score is selection-biased, not an independent test estimate. Improvement on hidden domains is not guaranteed.

Source: [scikit-learn text feature extraction](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
import joblib

OPT_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
OPT = ROOT / "results/jianing_optimization" / OPT_ID
OPT.mkdir(parents=True, exist_ok=False)
OPT_DATA = ROOT / "data/processed/jianing_optimization" / OPT_ID
OPT_DATA.mkdir(parents=True, exist_ok=False)
# Editable, bounded search: keep the split and preprocessing fixed when changing C.
OPT_C_GRID = [0.3, 1.0, 3.0]
LEXICAL_SPECS = {
    "TF": dict(TF_CONFIG),
    "IDF_uni": {"analyzer":"word", "lowercase":True, "ngram_range":[1,1],
        "min_df":5, "max_df":1.0, "norm":"l2", "sublinear_tf":False,
        "token_pattern":TF_CONFIG["token_pattern"]},
    "Word12": {"analyzer":"word", "lowercase":True, "ngram_range":[1,2],
        "min_df":3, "max_features":60000, "norm":"l2", "sublinear_tf":True,
        "token_pattern":TF_CONFIG["token_pattern"]},
    "Char35": {"analyzer":"char", "lowercase":False, "ngram_range":[3,5],
        "min_df":3, "max_features":80000, "norm":"l2", "sublinear_tf":True}
}
# Each block is unit-normalized before weighted concatenation and final L2 normalization.
OPT_CONFIGS = [
    {"name":"TF_emb025", "lexical":["TF"], "emb_weight":0.25},
    {"name":"TF_emb050", "lexical":["TF"], "emb_weight":0.5},
    {"name":"IDF_uni", "lexical":["IDF_uni"], "emb_weight":0.0},
    {"name":"IDF_uni_emb050", "lexical":["IDF_uni"], "emb_weight":0.5},
    {"name":"Word12", "lexical":["Word12"], "emb_weight":0.0},
    {"name":"Word12_emb050", "lexical":["Word12"], "emb_weight":0.5},
    {"name":"Char35", "lexical":["Char35"], "emb_weight":0.0},
    {"name":"WordChar", "lexical":["Word12","Char35"], "emb_weight":0.0},
    {"name":"WordChar_emb025", "lexical":["Word12","Char35"], "emb_weight":0.25},
    {"name":"WordChar_emb050", "lexical":["Word12","Char35"], "emb_weight":0.5}
]
assert len(OPT_CONFIGS)*len(OPT_C_GRID)==30
BASE_CONFIG = {"name":"Previous_TF_MiniLM", "lexical":["TF"], "emb_weight":1.0}

def make_lexical(kind):
    kwargs = dict(LEXICAL_SPECS[kind])
    kwargs["ngram_range"] = tuple(kwargs["ngram_range"])
    if kind=="TF":
        kwargs.pop("norm")
        return CountVectorizer(**kwargs)
    return TfidfVectorizer(**kwargs, dtype=np.float64)

def combine_blocks(blocks, embeddings, config):
    pieces = [blocks[k] for k in config["lexical"]]
    if config["emb_weight"]>0:
        pieces.append(sp.csr_matrix(embeddings)*config["emb_weight"])
    return normalize(sp.hstack(pieces,format="csr"),norm="l2",copy=False)

opt_vectors = {"TF": vectorizer}
opt_train_blocks = {"TF": X_tf_fit}
opt_val_blocks = {"TF": X_tf_val}
opt_prep = {"TF": {"train":tf_train_seconds, "validation":tf_val_seconds}}
feature_records = []
for kind in ["IDF_uni","Word12","Char35"]:
    extractor = make_lexical(kind)
    started=time.perf_counter(); a=extractor.fit_transform(fit.text)
    train_seconds=time.perf_counter()-started
    started=time.perf_counter(); b=extractor.transform(val.text)
    val_seconds=time.perf_counter()-started
    opt_vectors[kind]=extractor; opt_train_blocks[kind]=a; opt_val_blocks[kind]=b
    opt_prep[kind]={"train":train_seconds,"validation":val_seconds}
    feature_records.append({"representation":kind,"features":a.shape[1],
        "train_nonzero":a.nnz,"feature_train_seconds":train_seconds,"feature_val_seconds":val_seconds})
    print(f"Built {kind}: {a.shape[1]:,} features",flush=True)
feature_table=pd.DataFrame(feature_records)
display(feature_table.round(3))
feature_table.to_csv(OPT/"feature_summary.csv",index=False)
search_plan={"C_grid":OPT_C_GRID,"candidates":OPT_CONFIGS,"lexical_specs":LEXICAL_SPECS,
    "embedding_config":EMB_CONFIG,"svm_tol":TOL,"svm_max_iter":MAX_ITER,
    "seed":SEED,"data_sha256":DATA_SHA,"split_sha256":SPLIT_SHA,
    "source_baseline_run":RUN_ID,"baseline":{**BASE_CONFIG,"C":1.0}}
(OPT/"search_plan.json").write_text(json.dumps(search_plan,indent=2))

### 7B. Fit candidates and record every result
Each candidate changes an explicit feature configuration. C is tried on the same short grid. All trials are retained, including unsuccessful ones. SVM fitting time and feature preparation time are reported separately. A convergence failure stops the experiment. MiniLM representations are frozen and reused.

In [ ]:
opt_records=[]
opt_predictions=val[["id","label"]].rename(columns={"label":"true_label"}).copy()
opt_models={}

def fit_opt_svm(a,y,C):
    model=LinearSVC(C=C,dual="auto",max_iter=MAX_ITER,tol=TOL,random_state=SEED)
    started=time.perf_counter()
    with threadpool_limits(limits=4),warnings.catch_warnings():
        warnings.simplefilter("error",ConvergenceWarning)
        model.fit(a,y)
    return model,time.perf_counter()-started

for config in OPT_CONFIGS:
    start=time.perf_counter(); a=combine_blocks(opt_train_blocks,X_emb_fit,config)
    fusion_seconds=time.perf_counter()-start
    b=combine_blocks(opt_val_blocks,X_emb_val,config)
    prep=sum(opt_prep[k]["train"] for k in config["lexical"])+fusion_seconds
    if config["emb_weight"]>0: prep+=embedding_timing["train"]
    for C in OPT_C_GRID:
        model,seconds=fit_opt_svm(a,y_fit,C)
        s_train=model.decision_function(a);s_val=model.decision_function(b)
        train_auc=roc_auc_score(y_fit,s_train);val_auc=roc_auc_score(y_val,s_val)
        key=f"{config['name']}__C_{C:g}"
        opt_models[key]=model;opt_predictions[key]=s_val
        opt_records.append({"candidate":config["name"],"C":C,"features":a.shape[1],
            "embedding_weight":config["emb_weight"],"train_AUROC":train_auc,
            "validation_AUROC":val_auc,"gap":train_auc-val_auc,"svm_fit_seconds":seconds,
            "feature_train_seconds":prep,"train_pipeline_seconds":prep+seconds,
            "iterations":int(np.max(model.n_iter_))})
        print(f"{key}: train={train_auc:.6f}, validation={val_auc:.6f}",flush=True)
        # Save progress after every completed fit.
        pd.DataFrame(opt_records).to_csv(OPT/"all_trials.csv",index=False)
        opt_predictions.to_csv(OPT/"validation_scores.csv",index=False)
    del a,b
opt_table=pd.DataFrame(opt_records).sort_values(["validation_AUROC","features","C"],ascending=[False,True,True])
display(opt_table.round(6))
opt_table.to_csv(OPT/"ranked_trials.csv",index=False)

## 8. Compare with the previous best and check generalization
The original best (TF + MiniLM, C=1) remains the reference. Plot the best searched setting per new feature configuration, and summarize performance in the same length bins used in the discussion. Subgroup scores describe associations, not causes.

A paired bootstrap describes the observed validation difference only. Because the winner was selected on this validation set, its interval is not a selection-adjusted guarantee and says nothing directly about hidden Kaggle data.

In [ ]:
winner=opt_table.iloc[0]
winner_config=next(c.copy() for c in OPT_CONFIGS if c["name"]==winner.candidate)
winner_config["C"]=float(winner.C)
winner_key=f"{winner.candidate}__C_{winner.C:g}"
base_score=validation_scores["TF_MiniLM_fused__C_1__decision_score"].to_numpy()
base_auc=roc_auc_score(y_val,base_score)
best_score=opt_predictions[winner_key].to_numpy()
best_per_config=opt_table.drop_duplicates("candidate")
print(f"Previous best validation AUROC: {base_auc:.6f}")
print(f"New search winner: {winner_key}, AUROC={winner.validation_AUROC:.6f}, change={winner.validation_AUROC-base_auc:+.6f}")
fig,ax=plt.subplots(figsize=(7,5))
fpr,tpr,_=roc_curve(y_val,base_score);ax.plot(fpr,tpr,label=f"Previous TF+MiniLM: {base_auc:.4f}",linewidth=2)
# Top three distinct configurations keep the plot readable; all trials are in the table.
for row in best_per_config.head(3).itertuples():
    key=f"{row.candidate}__C_{row.C:g}"
    fpr,tpr,_=roc_curve(y_val,opt_predictions[key]);ax.plot(fpr,tpr,label=f"{row.candidate}, C={row.C:g}: {row.validation_AUROC:.4f}")
ax.plot([0,1],[0,1],"k--",linewidth=1)
ax.set(xlabel="False positive rate",ylabel="True positive rate",title="Validation ROC: optimization vs previous best")
ax.legend(fontsize=8);fig.tight_layout();fig.savefig(OPT/"optimization_roc.png",dpi=160);plt.show()
length_bin=pd.cut(val.text.str.split().str.len(),[0,150,250,np.inf],labels=["<=150","151-250",">250"])
subgroups=[]
for name in length_bin.cat.categories:
    ix=(length_bin==name).to_numpy()
    subgroups.append({"word_length":name,"count":int(ix.sum()),"human":int((y_val[ix]==0).sum()),
        "AI":int((y_val[ix]==1).sum()),"baseline_AUROC":roc_auc_score(y_val[ix],base_score[ix]),
        "new_AUROC":roc_auc_score(y_val[ix],best_score[ix])})
subgroup_table=pd.DataFrame(subgroups);display(subgroup_table.round(6));subgroup_table.to_csv(OPT/"length_subgroups.csv",index=False)
rng=np.random.default_rng(SEED);deltas=[]
for _ in range(1000):
    ix=rng.integers(0,len(y_val),len(y_val))
    deltas.append(roc_auc_score(y_val[ix],best_score[ix])-roc_auc_score(y_val[ix],base_score[ix]))
interval=np.quantile(deltas,[.025,.975]).tolist()
print("Descriptive paired-bootstrap 95% interval for validation gain:",interval)
# Save selected processed matrices and fitted artifacts locally; no test predictions.
best_train=combine_blocks(opt_train_blocks,X_emb_fit,winner_config)
best_val=combine_blocks(opt_val_blocks,X_emb_val,winner_config)
sp.save_npz(OPT_DATA/"selected_train.npz",best_train)
sp.save_npz(OPT_DATA/"selected_validation.npz",best_val)
fit[["id","label"]].to_csv(OPT_DATA/"train_rows.csv",index=False)
val[["id","label"]].to_csv(OPT_DATA/"validation_rows.csv",index=False)
joblib.dump({"extractors":{k:opt_vectors[k] for k in winner_config["lexical"]},
    "svm":opt_models[winner_key],"config":winner_config},OPT_DATA/"selected_model.joblib")
(OPT/"selected_config.json").write_text(json.dumps({"selected":winner_config,"search_plan":search_plan,
    "validation_AUROC":float(winner.validation_AUROC),"baseline_AUROC":base_auc,
    "descriptive_bootstrap_gain_95pct":interval,"selection_note":"Selected on this validation set; not an independent test estimate"},indent=2))

### 8B. Training-only three-fold stability check
Compare the previous baseline and the selected new configuration using the **same three folds inside the 8,000 training articles**. Refit every vocabulary and IDF within each fold. Frozen embeddings can be reused because they were generated independently of labels.

This checks whether the apparent gain is also present on other partitions of the supplied training distribution. The new configuration was selected using the separate validation set, so this is an internal stability diagnostic, not nested-CV performance of the entire search and not evidence of robustness to hidden domain shift. No further search is performed using these results.

In [ ]:
cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED)
cv_records=[];cv_ids=[]
checks=[{**BASE_CONFIG,"C":1.0},winner_config]
for fold,(a_idx,b_idx) in enumerate(cv.split(fit.text,y_fit),start=1):
    fold_train={};fold_eval={}
    required=set(k for config in checks for k in config["lexical"])
    started=time.perf_counter()
    for kind in sorted(required):
        ex=make_lexical(kind)
        fold_train[kind]=normalize(ex.fit_transform(fit.text.iloc[a_idx]))
        fold_eval[kind]=normalize(ex.transform(fit.text.iloc[b_idx]))
    for config in checks:
        a=combine_blocks(fold_train,X_emb_fit[a_idx],config)
        b=combine_blocks(fold_eval,X_emb_fit[b_idx],config)
        model,seconds=fit_opt_svm(a,y_fit[a_idx],config["C"])
        auc=roc_auc_score(y_fit[b_idx],model.decision_function(b))
        cv_records.append({"candidate":config["name"],"C":config["C"],"fold":fold,
                           "fold_train_rows":len(a_idx),"fold_eval_rows":len(b_idx),"AUROC":auc})
        print(f"Fold {fold}: {config['name']}, AUROC={auc:.6f}",flush=True)
    cv_ids.extend({"id":int(fit.id.iloc[i]),"evaluation_fold":fold} for i in b_idx)
    pd.DataFrame(cv_records).to_csv(OPT/"training_cv_folds.csv",index=False)
cv_table=pd.DataFrame(cv_records)
cv_summary=cv_table.groupby("candidate").AUROC.agg(["mean","std","min","max"]).reset_index()
display(cv_summary.round(6))
cv_summary.to_csv(OPT/"training_cv_summary.csv",index=False)
pd.DataFrame(cv_ids).to_csv(OPT/"training_cv_ids.csv",index=False)
(OPT.parent/"latest_run.txt").write_text(OPT_ID+"\n")
print("Saved optimization outputs to:",OPT.relative_to(ROOT))
print("The existing Kaggle CSV has not changed. A new probability-producing submission needs calibration and full-data refitting.")

## 9. Report Q2 — Effect of training data size
Fix Word+Character TF-IDF + chunked MiniLM (weight 0.25), Linear SVM C=1, and the shared 2,000-row validation partition. Use nested, balanced subsets of the 8,000 training rows, with sampling seed 42. Fit vocabulary and IDF again on each subset; never reuse vocabulary fitted on all 8,000 rows for a smaller subset. Frozen, label-independent MiniLM embeddings may be reused after checking row IDs and the data hash.

This is one sampling realization per size, not an uncertainty estimate. The configuration was previously selected on this validation set, so this is a descriptive learning curve rather than a new independent evaluation. No calibration is needed for AUROC based on SVM decision scores. Run the preceding embedding step first on a fresh machine.

In [ ]:
from pathlib import Path
import json, hashlib, time, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.svm import LinearSVC
from sklearn.metrics import roc_auc_score
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits

LC_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "data/raw/train.csv").exists() and (p / "notebooks").exists())
LC_CACHE = LC_ROOT / "data/processed/jianing_svm"
LC_OUT = LC_ROOT / "results/jianing_learning_curve"
LC_OUT.mkdir(parents=True, exist_ok=True)
lc_df = pd.read_csv(LC_ROOT / "data/raw/train.csv").set_index("id", drop=False)
lc_split = pd.read_csv(LC_ROOT / "data/splits/seed42_stratified_8000_2000.csv")
lc_fit_ids = lc_split.loc[lc_split.partition == "train", "id"].to_numpy()
lc_val_ids = lc_split.loc[lc_split.partition == "validation", "id"].to_numpy()
lc_fit = lc_df.loc[lc_fit_ids].reset_index(drop=True)
lc_val = lc_df.loc[lc_val_ids].reset_index(drop=True)
assert len(lc_fit)==8000 and len(lc_val)==2000
assert set(lc_fit_ids).isdisjoint(lc_val_ids)
for name, ids in [("train", lc_fit_ids), ("validation", lc_val_ids)]:
    assert np.array_equal(pd.read_csv(LC_CACHE / f"{name}_rows.csv").id.to_numpy(), ids)
lc_meta = json.loads((LC_CACHE / "minilm_cache.json").read_text())
lc_hash = hashlib.sha256((LC_ROOT / "data/raw/train.csv").read_bytes()).hexdigest()
# Check the original cache manifest against the same dataset.
lc_cache_spec = dict(data_sha=lc_hash, train_ids=lc_fit_ids.tolist(),
    val_ids=lc_val_ids.tolist(), embedding=lc_meta["config"], versions=lc_meta["versions"])
assert hashlib.sha256(json.dumps(lc_cache_spec,sort_keys=True).encode()).hexdigest()==lc_meta["cache_key"], "Embedding cache mismatch."
lc_emb_fit = np.load(LC_CACHE / "minilm_train.npy", allow_pickle=False)
lc_emb_val = np.load(LC_CACHE / "minilm_validation.npy", allow_pickle=False)
assert lc_emb_fit.shape==(8000,384) and lc_emb_val.shape==(2000,384)
lc_rng = np.random.default_rng(42)
lc_orders = {k: lc_rng.permutation(np.flatnonzero(lc_fit.label.to_numpy()==k)) for k in [0,1]}
lc_sizes = [500,1000,2000,5000,8000]
lc_rows, lc_ids, lc_scores = [], [], []
lc_specs = {
    "word": dict(analyzer="word",lowercase=True,ngram_range=(1,2),min_df=3,
                 max_features=60000,sublinear_tf=True,norm="l2",token_pattern=r"(?u)\b\w\w+\b"),
    "char": dict(analyzer="char",lowercase=False,ngram_range=(3,5),min_df=3,
                 max_features=80000,sublinear_tf=True,norm="l2")}
for size in lc_sizes:
    idx = np.sort(np.concatenate([lc_orders[k][:size//2] for k in [0,1]]))
    subset = lc_fit.iloc[idx]
    assert subset.label.value_counts().to_dict()=={0:size//2,1:size//2}
    started = time.perf_counter()
    train_blocks, valid_blocks = [], []
    for spec in lc_specs.values():
        extractor = TfidfVectorizer(**spec, dtype=np.float64)
        train_blocks.append(extractor.fit_transform(subset.text))
        valid_blocks.append(extractor.transform(lc_val.text))
    train_blocks.append(sp.csr_matrix(lc_emb_fit[idx])*0.25)
    valid_blocks.append(sp.csr_matrix(lc_emb_val)*0.25)
    a = normalize(sp.hstack(train_blocks,format="csr"),norm="l2")
    b = normalize(sp.hstack(valid_blocks,format="csr"),norm="l2")
    feature_seconds = time.perf_counter()-started
    model = LinearSVC(C=1.0,dual="auto",max_iter=50000,tol=1e-5,random_state=42)
    started = time.perf_counter()
    with threadpool_limits(limits=4), warnings.catch_warnings():
        warnings.simplefilter("error",ConvergenceWarning)
        model.fit(a,subset.label)
    fit_seconds = time.perf_counter()-started
    train_score, valid_score = model.decision_function(a), model.decision_function(b)
    lc_rows.append(dict(training_rows=size,train_AUROC=roc_auc_score(subset.label,train_score),
        validation_AUROC=roc_auc_score(lc_val.label,valid_score),features=a.shape[1],
        lexical_and_fusion_seconds=feature_seconds,svm_fit_seconds=fit_seconds))
    lc_ids.extend(dict(training_rows=size,id=int(i)) for i in subset.id)
    lc_scores.extend(dict(training_rows=size,id=int(i),label=int(y),score=float(s))
                     for i,y,s in zip(lc_val.id,lc_val.label,valid_score))
    pd.DataFrame(lc_rows).to_csv(LC_OUT/"learning_curve.csv",index=False)
    print(f"n={size}: train AUROC={lc_rows[-1]['train_AUROC']:.6f}, validation AUROC={lc_rows[-1]['validation_AUROC']:.6f}",flush=True)
    del a,b,train_blocks,valid_blocks
lc_table=pd.DataFrame(lc_rows)
display(lc_table.round(6))
pd.DataFrame(lc_ids).to_csv(LC_OUT/"subset_ids.csv",index=False)
pd.DataFrame(lc_scores).to_csv(LC_OUT/"validation_scores.csv",index=False)
(LC_OUT/"config.json").write_text(json.dumps(dict(seed=42,sizes=lc_sizes,C=1.0,
    embedding_weight=0.25,lexical_specs=lc_specs,data_sha256=lc_hash,
    sampling="nested balanced subsets; one realization",validation_rows=2000,
    timing="excludes cached frozen MiniLM extraction",versions={"numpy":np.__version__}),indent=2))
fig,ax=plt.subplots(figsize=(7,4.3))
ax.plot(lc_table.training_rows,lc_table.train_AUROC,"o-",label="Training AUROC")
ax.plot(lc_table.training_rows,lc_table.validation_AUROC,"o-",label="Validation AUROC")
ax.set(xlabel="Number of labeled training examples",ylabel="AUROC",ylim=(0.5,1.01),
       title="Learning curve: Word + Character TF-IDF + MiniLM, SVM C=1")
ax.set_xticks(lc_sizes);ax.grid(alpha=0.25);ax.legend();fig.tight_layout()
fig.savefig(LC_OUT/"learning_curve.png",dpi=200)
fig.savefig(LC_OUT/"learning_curve.pdf")
plt.show()
print("Saved learning-curve results:",LC_OUT)
